# Configurações Iniciais

## Realizando import das bibliotecas que serão utilizadas 

In [1]:
import pandas as pd
from dotenv import load_dotenv
import boto3

## Carregando variáveis de ambiente

In [2]:
load_dotenv()

True

## Lendo Dataset

In [3]:
s3_get_bucket = boto3.client('s3')
obj = s3_get_bucket.get_object(
    Bucket='desafio-sprint-06',
    Key='comprasGOV.csv'
)

df = pd.read_csv(obj['Body'])

# Análises

## 5 Itens mais caros, a mediana e média de todas as contratações efetivadas

In [4]:
analise_1 = df.copy()

### Função de conversão

In [5]:
analise_1['valor_total_resultado'] = pd.to_numeric(
    analise_1['valor_total_resultado'],
    errors='coerce'
)

### Ordenando Dataset

In [6]:
analise_1 = analise_1.sort_values('valor_total_resultado', ascending=False)

### Função de agregação

In [7]:
media = analise_1['valor_total_resultado'].mean()
mediana = analise_1['valor_total_resultado'].median()
print(f"Media: R${round(media, 2)}\nMediana: R${round(mediana, 2)}")

Media: R$1939.11
Mediana: R$1170.88


### Filtrando os 5 Itens mais caros

In [8]:
analise_1 = analise_1.head(5)

## Itens Homologados que foram comprados em 2025

In [9]:
analise_2 = df.copy()

### Função de data

In [10]:
analise_2['ano_compra'] = pd.to_datetime(analise_2['ano_compra'], format="%Y")
analise_2['ano_compra'] = analise_2['ano_compra'].dt.year

### Cláusula que filtra dados usando ao menos dois operadores lógicos

In [11]:
analise_2 = analise_2.loc[
    (analise_2['ano_compra'] == 2025)
    & (analise_2['valor_total_resultado'].notna())
]

## Adição da coluna categoria_item

In [12]:
analise_3 = df.copy()

### Inserindo coluna e atribuindo valor default

In [13]:
analise_3.loc[:,'categoria_item'] = "Outros"

### Função de string

In [14]:
mascara = analise_3['descricao_resumida'].str.contains

### Função condicional

In [15]:
analise_3.loc[mascara("fruta|legum|verdura", case=False, na=False), 'categoria_item'] = "Hortifruti"
analise_3.loc[mascara("leite|creme de leite|manteiga|iogurte", case=False, na=False), 'categoria_item'] = "Laticínios"
analise_3.loc[mascara("arroz|feijão|café|farinha|macarrão|açúcar|sal|fermento", case=False, na=False), 'categoria_item'] = "Grãos e Mercearia"
analise_3.loc[mascara("embutido|polpa|conserva", case=False, na=False), 'categoria_item'] = "Processados"
analise_3.loc[mascara("carne|frango|ave|bovina|suína|peixe", case=False, na=False), 'categoria_item'] = "Carnes"

# Salvando os arquivos resultantes das análises no bucket

## Salvando Localmente

In [16]:
for i, dataset in enumerate([analise_1, analise_2, analise_3], 1):
    dataset.to_csv(f'analise-{i}.csv', index=False)

## Salvando no Bucket

In [17]:
s3_send_to_bucket = boto3.resource('s3')
bucket = s3_send_to_bucket.Bucket('desafio-sprint-06')
for num in range(1, 4):
    bucket.upload_file(f'analise-{num}.csv', f'analise-{num}.csv')